# Task 4 — Open-Set Recognition (CIFAR-10 known; CIFAR-100 near / far unknowns)

* CIFAR ResNet-18 (3×3 stride-1 stem, no max-pool), 32×32 inputs, random init for Vanilla / GCSC / RPL.
* Recipe: random crop 32 (pad 4) + flip, SGD (lr 0.1, momentum 0.9, wd 5e-4), cosine decay, batch 128, 100 epochs, seed 6304; checkpoint = best CIFAR-10 validation accuracy (stratified 90/10 split of the official training set, seed 6304).
* **GCSC** = identical recipe + `RandAugment(num_ops=2, magnitude=9)` after crop/flip.
* **PROSER** = initialized from the selected Vanilla checkpoint + 5 dummy classifiers; 50 epochs, SGD lr 1e-3 (momentum 0.9, wd 5e-4, cosine), batch 128. First half of each batch: classifier-placeholder loss (β = 1); second half: manifold mixup after `layer2` between different-class pairs, λ ~ Beta(2,2), trained towards the dummy slot (γ = 0.1). Dummy logits are collapsed with a max (paper's f̂ = [f, max_k ŵ_k]).
* **RPL** (optional extension, enabled): one learnable reciprocal point per class; logits = mean squared distance to each point; open-space term MSE(distance to own point, learnable radius R) with weight 0.1; same recipe as Vanilla.
* **Unknowns (CIFAR-100 test only):** near = bus, pickup_truck, motorcycle, tractor, wolf, fox, leopard, camel; far = bottle, bowl, chair, clock, keyboard, mushroom, sunflower, wardrobe (800 each). They are loaded only after all checkpoints and score definitions are fixed.
* Rejection: τ = 95th percentile of u(x) on CIFAR-10 validation; accept iff u(x) ≤ τ.
* **Changes after run v1 (PROSER only):** (1) the mixed layer2 outputs pass through layer3/layer4 without updating those BatchNorm layers' running statistics (momentum 0 for that pass; they still normalize with the batch's own statistics, as in the reference). In run v1 the mixed batches also updated them; CIFAR-10 validation accuracy fell from 0.948 to 0.886 within four epochs, and the checkpoint rule then kept epoch 0, before the data placeholders had been learned. An intermediate PROSER run (the `v2_*` diagnostics files) used eval-mode BatchNorm for that pass instead, which made training erratic (validation accuracy 0.72–0.94, losses rising in the last epochs at a learning rate of ~1e-6) and was replaced. (2) The final-epoch model is saved as well and reported as a clearly marked supplementary row (`proser_last_*`), fixed before evaluation. The main PROSER rows still use the best CIFAR-10 validation accuracy. The v1 and v2 logs are in `task4/results/diagnostics/`.

Set `QUICK_RUN = True` for a 2-epoch dry run on subsets (never report those numbers).

In [1]:
import sys, json
from pathlib import Path

ROOT = Path.cwd().resolve()
while not (ROOT / "common").is_dir():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

from common.seed import set_seed
from common.paths import DATA_ROOT, results_dir, checkpoints_dir, cache_dir, figure_path
from common.logging import save_table, save_json
from common.plotting import save_figure

SEED = 6304
QUICK_RUN = False
RETRAIN = False
RUN_RPL = True            # optional extension (Step 5)
NUM_WORKERS = 4
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
RESULTS, CKPT = results_dir(4), checkpoints_dir(4)
CACHE = cache_dir(4) / ("quick" if QUICK_RUN else "full"); CACHE.mkdir(parents=True, exist_ok=True)
if QUICK_RUN:
    CKPT = CKPT / "quick"; CKPT.mkdir(exist_ok=True)
set_seed(SEED)
pd.set_option("display.precision", 4); pd.set_option("display.width", 220)
METHODS = ["vanilla", "gcsc", "proser", "proser_last"] + (["rpl"] if RUN_RPL else [])  # proser_last = final-epoch PROSER (supplementary)

In [2]:
from task4.data.cifar_source import load_cifar
from task4.data.make_splits import save_cifar10_split
labels_c10 = np.array(load_cifar(DATA_ROOT, "cifar10", True).targets)
save_cifar10_split(labels_c10, RESULTS / "cifar10_split_seed6304.json", seed=SEED)

## Steps 1, 3, 4, 5 — Training (CIFAR-10 only)

In [3]:
from task4.pipeline import train_or_load
trained, summaries = {}, {}
for m in METHODS:   # vanilla must come before proser
    ext, extra, summ = train_or_load(m, DATA_ROOT, CKPT, RESULTS, DEVICE, NUM_WORKERS, retrain=RETRAIN, quick=QUICK_RUN, seed=SEED)
    trained[m], summaries[m] = (ext, extra), summ
display(save_table(list(summaries.values()), RESULTS / "training_summary"))

[vanilla] loaded /workspace/pa1-beyond-iid/task4/checkpoints/vanilla.pth


[gcsc] loaded /workspace/pa1-beyond-iid/task4/checkpoints/gcsc.pth


[proser_last] loaded /workspace/pa1-beyond-iid/task4/checkpoints/proser_last.pth


[rpl] loaded /workspace/pa1-beyond-iid/task4/checkpoints/rpl.pth


,method,best_val_acc,epochs,best_epoch,epoch,final_val_acc
0,vanilla,0.9536,100,97.0,NaN,NaN
1,gcsc,0.9586,100,97.0,NaN,NaN
2,proser,0.9534,50,21.0,NaN,NaN
3,proser_last,NaN,50,NaN,49.0,0.9524
4,rpl,0.9562,100,99.0,NaN,NaN


In [4]:
# Training curves: loss and CIFAR-10 validation accuracy per epoch.
TRAINED = [m for m in METHODS if m != "proser_last"]  # proser_last shares PROSER's log
fig, axes = plt.subplots(1, len(TRAINED), figsize=(3.2 * len(TRAINED), 2.9), squeeze=False)
for ax, m in zip(axes[0], TRAINED):
    path = RESULTS / f"task4_{m}.metrics.jsonl"
    h = [json.loads(l) for l in path.read_text().splitlines() if l.strip()] if path.exists() else []
    if not h:
        ax.set_title(f"{m}: no log"); continue
    ep = [r["step"] for r in h]
    for k in ["train_loss", "standard_ce", "classifier_placeholder_ce", "data_placeholder_ce", "cls_loss", "open_space_loss"]:
        if k in h[0]:
            ax.plot(ep, [r[k] for r in h], label=k, lw=1)
    ax2 = ax.twinx(); ax2.plot(ep, [r["val_acc"] for r in h], color="k", lw=1.2, label="val acc"); ax2.set_ylim(0, 1)
    ax.set_title(m, fontsize=9); ax.set_xlabel("epoch"); ax.legend(fontsize=5, loc="center right")
fig.tight_layout()
save_figure(fig, figure_path("task4_training_curves.png"))

PosixPath('/workspace/pa1-beyond-iid/report/figures/task4_training_curves.png')

## Configuration lock
Checkpoints are fixed; the score definitions below (MSP, MLS, Energy, Mahalanobis, PROSER placeholder, RPL distance) and the validation-only 95th-percentile threshold rule are fixed before any CIFAR-100 image is loaded.

In [5]:
save_json({"summaries": summaries, "scores": ["msp", "mls", "energy", "mahalanobis", "proser_placeholder",
                                               "proser_dummy_prob", "rpl_softmax_max", "rpl_max_distance"],
           "threshold": "95th percentile of u on CIFAR-10 validation; accept iff u <= tau"}, RESULTS / "locked_configuration.json")

PosixPath('/workspace/pa1-beyond-iid/task4/results/locked_configuration.json')

## Extract features / logits once per model (CIFAR-10 train [unaugmented], val, test; CIFAR-100 near, far)
All scores of a model are computed from exactly these saved arrays (`task4/cache/`).

In [6]:
from task4.pipeline import eval_loaders, extract_all
loaders = eval_loaders(DATA_ROOT, NUM_WORKERS, quick=QUICK_RUN)
outs = {m: extract_all(m, *trained[m], loaders, DEVICE, CACHE, overwrite=RETRAIN) for m in METHODS}
csa = {m: float((outs[m]["test"]["logits"].argmax(1) == outs[m]["test"]["labels"]).mean()) for m in METHODS}
print("CIFAR-10 test accuracy (known-class logits only):", csa)

CIFAR-10 test accuracy (known-class logits only): {'vanilla': 0.9494, 'gcsc': 0.9521, 'proser': 0.9474, 'proser_last': 0.9497, 'rpl': 0.949}


## Step 2 — Post-hoc scores on the frozen Vanilla model
u_MSP = 1 − max softmax; u_MLS = −max logit; u_Energy = −logsumexp(logits); u_Mah = min_c (f−μ_c)ᵀΣ⁻¹(f−μ_c) with class means and one shared diagonal covariance (+1e-6) from unaugmented CIFAR-10 *training* features.

In [7]:
from task4.scores.msp import msp_score
from task4.scores.mls import mls_score
from task4.scores.energy import energy_score
from task4.scores.mahalanobis import MahalanobisScorer
from task4.evaluation.metrics import osr_row

def posthoc_scores(o):
    mah = MahalanobisScorer().fit(o["train"]["features"], o["train"]["labels"], n_classes=10)
    return {s: {split: fn(o[split]) for split in ["val", "test", "near", "far"]} for s, fn in {
        "msp": lambda d: msp_score(d["logits"]), "mls": lambda d: mls_score(d["logits"]),
        "energy": lambda d: energy_score(d["logits"]), "mahalanobis": lambda d: mah.score(d["features"])}.items()}

U = {m: posthoc_scores(outs[m]) for m in ["vanilla", "gcsc", "proser", "proser_last"]}
step2 = [osr_row(s, **{f"u_{k}": U["vanilla"][s][k] for k in ["val", "test", "near", "far"]}) for s in ["msp", "mls", "energy", "mahalanobis"]]
display(save_table(step2, RESULTS / "step2_vanilla_scores"))

,name,auroc_near,auroc_far,auroc_all,tau,val_acceptance,test_known_acceptance,near_rejection,far_rejection,all_rejection,fpr95_near,fpr95_far,fpr95_all
0,msp,0.8213,0.9002,0.8608,0.1536,0.95,0.9477,0.3075,0.4700,0.3887,0.6925,0.5300,0.6112
1,mls,0.8058,0.8993,0.8526,-5.8614,0.95,0.9474,0.3425,0.5687,0.4556,0.6575,0.4313,0.5444
2,energy,0.8059,0.9000,0.8530,-6.0263,0.95,0.9491,0.3325,0.5737,0.4531,0.6675,0.4263,0.5469
3,mahalanobis,0.8094,0.9210,0.8652,3019.0715,0.95,0.9468,0.2963,0.5450,0.4206,0.7037,0.4550,0.5794


In [8]:
# Compact figure: score distributions (known test / near / far, with tau) and ROC curves for MSP, MLS, Mahalanobis.
from sklearn.metrics import roc_curve
titles = {"msp": "MSP: 1 − max p", "mls": "MLS: −max z", "mahalanobis": "Mahalanobis (min over classes)"}
fig, axes = plt.subplots(2, 3, figsize=(11, 5.6))
for j, s in enumerate(["msp", "mls", "mahalanobis"]):
    u = U["vanilla"][s]
    tau = np.percentile(u["val"], 95)
    ax = axes[0, j]
    lo, hi = np.percentile(np.concatenate([u["test"], u["near"], u["far"]]), [0.5, 99.5])
    bins = np.linspace(lo, hi, 60)
    for k, lab, c in [("test", "CIFAR-10 test (known)", "C0"), ("near", "near unknown", "C3"), ("far", "far unknown", "C2")]:
        ax.hist(np.clip(u[k], lo, hi), bins=bins, density=True, histtype="stepfilled", alpha=0.35, color=c, label=lab)
    ax.axvline(tau, color="k", ls="--", lw=1, label="τ (val 95th pct.)")
    ax.set_title(titles[s], fontsize=9); ax.set_yticks([])
    if s == "msp":
        ax.set_yscale("log")
    ax = axes[1, j]
    for k, c in [("near", "C3"), ("far", "C2")]:
        y_ = np.r_[np.zeros(len(u["test"])), np.ones(len(u[k]))]
        fpr, tpr, _ = roc_curve(y_, np.r_[u["test"], u[k]])
        r = next(r for r in step2 if r["name"] == s)
        ax.plot(fpr, tpr, color=c, label=f"known vs {k} (AUROC {r['auroc_' + k]:.3f})")
    ax.plot([0, 1], [0, 1], "k:", lw=0.8)
    ax.set_xlabel("FPR (known flagged unknown)"); ax.set_ylabel("TPR (unknown flagged)"); ax.legend(fontsize=7)
axes[0, 0].legend(fontsize=6)
fig.tight_layout()
save_figure(fig, figure_path("task4_score_distributions_roc.png"))

PosixPath('/workspace/pa1-beyond-iid/report/figures/task4_score_distributions_roc.png')

In [9]:
# Where do the scores agree / disagree? Rank correlations and decision disagreements on unknowns.
from scipy.stats import spearmanr
names = ["msp", "mls", "energy", "mahalanobis"]
pool = {s: np.concatenate([U["vanilla"][s][k] for k in ["test", "near", "far"]]) for s in names}
corr = pd.DataFrame([[spearmanr(pool[a], pool[b])[0] for b in names] for a in names], index=names, columns=names)
display(corr); corr.to_csv(RESULTS / "step2_score_spearman.csv")
dis = []
for g in ["near", "far", "test"]:
    rej = {s: U["vanilla"][s][g] > np.percentile(U["vanilla"][s]["val"], 95) for s in names}
    for a in names:
        for b in names:
            if a < b:
                dis.append({"set": g, "a": a, "b": b, "rejected_by_a_only": float((rej[a] & ~rej[b]).mean()),
                            "rejected_by_b_only": float((rej[b] & ~rej[a]).mean()), "both": float((rej[a] & rej[b]).mean())})
display(save_table(dis, RESULTS / "step2_score_decision_disagreement"))

,msp,mls,energy,mahalanobis
msp,1.0000,0.9610,0.9597,0.7002
mls,0.9610,1.0000,0.9999,0.5895
energy,0.9597,0.9999,1.0000,0.5880
mahalanobis,0.7002,0.5895,0.5880,1.0000


,set,a,b,rejected_by_a_only,rejected_by_b_only,both
0,near,mls,msp,0.0838,0.0488,0.2587
1,near,energy,msp,0.0925,0.0675,0.2400
2,near,energy,mls,0.0088,0.0187,0.3237
3,near,energy,mahalanobis,0.0975,0.0612,0.2350
4,near,mahalanobis,msp,0.0350,0.0462,0.2612
5,near,mahalanobis,mls,0.0488,0.0950,0.2475
6,far,mls,msp,0.1425,0.0437,0.4263
7,far,energy,msp,0.1575,0.0537,0.4163
8,far,energy,mls,0.0150,0.0100,0.5587
9,far,energy,mahalanobis,0.1100,0.0813,0.4637


## Step 6 — Model comparison (MLS as the common score; PROSER placeholder score as extra row)

In [10]:
from task4.methods.proser import placeholder_scores_from_logits
rows = [osr_row(f"{m}_mls", csa=csa[m], **{f"u_{k}": U[m]["mls"][k] for k in ["val", "test", "near", "far"]})
        for m in ["vanilla", "gcsc", "proser"]]
ph = {k: placeholder_scores_from_logits(outs["proser"][k]["logits"], outs["proser"][k]["extra"]) for k in ["val", "test", "near", "far"]}
for s in ["proser_placeholder", "proser_dummy_prob"]:
    rows.append(osr_row(s, csa=csa["proser"], **{f"u_{k}": ph[k][s] for k in ["val", "test", "near", "far"]}))
if RUN_RPL:
    # RPL rejects on the reciprocal-point distances. Two variants are reported:
    # the reference implementation's softmax-over-distances score, and its MLS
    # analogue (-max distance). Neither was chosen by looking at unknown data.
    from task4.methods.rpl import rpl_scores_from_distances
    rpl_u = {k: rpl_scores_from_distances(outs["rpl"][k]["logits"]) for k in ["val", "test", "near", "far"]}
    for s_name in ["rpl_softmax_max", "rpl_max_distance"]:
        rows.append(osr_row(s_name, csa=csa["rpl"], **{f"u_{k}": rpl_u[k][s_name] for k in ["val", "test", "near", "far"]}))
# Supplementary (fixed before evaluation): final-epoch PROSER, same scores.
rows.append(osr_row("proser_last_mls", csa=csa["proser_last"],
                    **{f"u_{k}": U["proser_last"]["mls"][k] for k in ["val", "test", "near", "far"]}))
ph_last = {k: placeholder_scores_from_logits(outs["proser_last"][k]["logits"], outs["proser_last"][k]["extra"])
           for k in ["val", "test", "near", "far"]}
for s in ["proser_placeholder", "proser_dummy_prob"]:
    rows.append(osr_row(s.replace("proser", "proser_last"), csa=csa["proser_last"],
                        **{f"u_{k}": ph_last[k][s] for k in ["val", "test", "near", "far"]}))
comp = save_table(rows, RESULTS / "step6_model_comparison")
display(comp)

# Supplementary: every post-hoc score on every model (same saved outputs).
supp = [osr_row(f"{m}_{s}", csa=csa[m], **{f"u_{k}": U[m][s][k] for k in ["val", "test", "near", "far"]})
        for m in ["vanilla", "gcsc", "proser", "proser_last"] for s in names]
display(save_table(supp, RESULTS / "step6_all_scores_all_models")[["name", "closed_set_accuracy", "auroc_near", "auroc_far", "near_rejection", "far_rejection"]])

# Per-class closed-set accuracy
from task4.pipeline import CIFAR10_CLASSES
pcs = []
for m in METHODS:
    t = outs[m]["test"]
    for k, c in enumerate(CIFAR10_CLASSES):
        pcs.append({"model": m, "class": c, "accuracy": float((t["logits"].argmax(1)[t["labels"] == k] == k).mean())})
display(save_table(pcs, RESULTS / "step6_per_class_csa").pivot(index="class", columns="model", values="accuracy"))

,name,closed_set_accuracy,auroc_near,auroc_far,auroc_all,tau,val_acceptance,test_known_acceptance,near_rejection,far_rejection,all_rejection,fpr95_near,fpr95_far,fpr95_all
0,vanilla_mls,0.9494,0.8058,0.8993,0.8526,-5.8614,0.95,0.9474,0.3425,0.5687,0.4556,0.6575,0.4313,0.5444
1,gcsc_mls,0.9521,0.8135,0.9004,0.8570,-6.4154,0.95,0.9460,0.3862,0.5913,0.4888,0.6138,0.4088,0.5112
2,proser_mls,0.9474,0.8124,0.9152,0.8638,-6.8790,0.95,0.9457,0.3400,0.5737,0.4569,0.6600,0.4263,0.5431
3,proser_placeholder,0.9474,0.7968,0.9071,0.8520,-1.2311,0.95,0.9445,0.3175,0.5300,0.4238,0.6825,0.4700,0.5763
4,proser_dummy_prob,0.9474,0.7970,0.9075,0.8522,0.2094,0.95,0.9439,0.3237,0.5350,0.4294,0.6763,0.4650,0.5706
5,rpl_softmax_max,0.9490,0.7599,0.8830,0.8214,0.4054,0.95,0.9449,0.3250,0.4913,0.4081,0.6750,0.5088,0.5919
6,rpl_max_distance,0.9490,0.5823,0.6387,0.6105,-10.3508,0.95,0.9445,0.2512,0.3588,0.3050,0.7488,0.6412,0.6950
7,proser_last_mls,0.9497,0.8242,0.9147,0.8695,-7.1719,0.95,0.9475,0.3475,0.5863,0.4669,0.6525,0.4138,0.5331
8,proser_last_placeholder,0.9497,0.8081,0.9096,0.8588,-1.3996,0.95,0.9453,0.3275,0.5625,0.4450,0.6725,0.4375,0.5550
9,proser_last_dummy_prob,0.9497,0.8083,0.9102,0.8593,0.1877,0.95,0.9455,0.3312,0.5675,0.4494,0.6687,0.4325,0.5506


,name,closed_set_accuracy,auroc_near,auroc_far,near_rejection,far_rejection
0,vanilla_msp,0.9494,0.8213,0.9002,0.3075,0.4700
1,vanilla_mls,0.9494,0.8058,0.8993,0.3425,0.5687
2,vanilla_energy,0.9494,0.8059,0.9000,0.3325,0.5737
3,vanilla_mahalanobis,0.9494,0.8094,0.9210,0.2963,0.5450
4,gcsc_msp,0.9521,0.8308,0.9062,0.3075,0.4462
5,gcsc_mls,0.9521,0.8135,0.9004,0.3862,0.5913
6,gcsc_energy,0.9521,0.8128,0.9005,0.3837,0.5988
7,gcsc_mahalanobis,0.9521,0.7895,0.9142,0.2612,0.5188
8,proser_msp,0.9474,0.8158,0.9098,0.3038,0.4763
9,proser_mls,0.9474,0.8124,0.9152,0.3400,0.5737


model,gcsc,proser,proser_last,rpl,vanilla
class,,,,,
airplane,0.960,0.956,0.964,0.953,0.965
automobile,0.980,0.985,0.983,0.979,0.981
bird,0.941,0.921,0.922,0.924,0.936
cat,0.889,0.864,0.875,0.886,0.874
deer,0.967,0.967,0.963,0.964,0.970
dog,0.923,0.916,0.922,0.932,0.915
frog,0.969,0.968,0.970,0.970,0.965
horse,0.962,0.964,0.962,0.954,0.956
ship,0.964,0.966,0.973,0.958,0.966


In [11]:
fig, axes = plt.subplots(1, 3, figsize=(11, 3.0))
lab = list(comp.name)
x = np.arange(len(lab))
axes[0].bar(x, comp.closed_set_accuracy, color="gray"); axes[0].set_ylim(comp.closed_set_accuracy.min() - 0.03, 1)
axes[0].set_title("Closed-set accuracy (CIFAR-10 test)", fontsize=9)
for ax, (a, b, t) in zip(axes[1:], [("auroc_near", "auroc_far", "AUROC"), ("near_rejection", "far_rejection", "Rejection @ τ (95% val acceptance)")]):
    ax.bar(x - 0.2, comp[a], 0.4, label="near", color="C3"); ax.bar(x + 0.2, comp[b], 0.4, label="far", color="C2")
    ax.set_title(t, fontsize=9); ax.legend(fontsize=7)
for ax in axes:
    ax.set_xticks(x, lab, rotation=30, ha="right", fontsize=7); ax.grid(axis="y", alpha=0.3)
fig.tight_layout()
save_figure(fig, figure_path("task4_model_comparison.png"))

PosixPath('/workspace/pa1-beyond-iid/report/figures/task4_model_comparison.png')

## Failure analysis (Vanilla, MLS threshold) — for analysis only, nothing here feeds back into any model, score or threshold

In [12]:
from task4.data.cifar100_unknowns import NEAR_UNKNOWN_CLASSES, FAR_UNKNOWN_CLASSES
from task4.evaluation.failure_analysis import acceptance_by_unknown_class, most_confident_accepted
c100 = load_cifar(DATA_ROOT, "cifar100", False)
fine_names = c100.classes
targets = np.array(c100.targets)
raw_idx = {"near": np.where(np.isin(targets, [fine_names.index(c) for c in NEAR_UNKNOWN_CLASSES]))[0],
           "far": np.where(np.isin(targets, [fine_names.index(c) for c in FAR_UNKNOWN_CLASSES]))[0]}
if QUICK_RUN:
    raw_idx = {k: v[::20] for k, v in raw_idx.items()}
tau = float(np.percentile(U["vanilla"]["mls"]["val"], 95))
acc_rows, ex_rows = [], []
for g in ["near", "far"]:
    o = outs["vanilla"][g]
    assert np.array_equal(o["labels"], targets[raw_idx[g]])
    u, preds = U["vanilla"]["mls"][g], o["logits"].argmax(1)
    acc_rows += acceptance_by_unknown_class(u, tau, preds, o["labels"], fine_names, g)
    ex_rows += most_confident_accepted(u, tau, preds, o["labels"], fine_names, g, n=8)
acc_df = save_table(acc_rows, RESULTS / "step6_vanilla_mls_acceptance_by_class")
display(acc_df[["group", "unknown_class", "accepted_frac", "top_absorbing_known"]])
ex_df = save_table(ex_rows, RESULTS / "step6_vanilla_mls_failure_examples")
display(ex_df)

# Heatmap: fraction of each unknown class accepted as each CIFAR-10 class.
mat = acc_df[[f"to_{c}" for c in CIFAR10_CLASSES]].values / acc_df["n"].values[:, None]
fig, ax = plt.subplots(figsize=(6.4, 5.2))
im = ax.imshow(mat, cmap="Reds", vmin=0, vmax=max(mat.max(), 1e-6))
ax.set_xticks(range(10), CIFAR10_CLASSES, rotation=45, ha="right", fontsize=7)
ax.set_yticks(range(len(acc_df)), [f"{r.unknown_class} ({r.group})" for r in acc_df.itertuples()], fontsize=7)
ax.axhline(len(NEAR_UNKNOWN_CLASSES) - 0.5, color="k", lw=1)
for i in range(mat.shape[0]):
    for k in range(10):
        if mat[i, k] >= 0.05:
            ax.text(k, i, f"{mat[i, k]:.2f}", ha="center", va="center", fontsize=5)
ax.set_title(f"Vanilla + MLS: unknowns accepted (u ≤ τ={tau:.2f}), by predicted class", fontsize=8)
fig.colorbar(im, fraction=0.03); fig.tight_layout()
save_figure(fig, figure_path("task4_acceptance_heatmap.png"))

# Image grid of the most confidently accepted near / far unknowns.
fig, axes = plt.subplots(2, 8, figsize=(12, 3.8))
for r, g in enumerate(["near", "far"]):
    sub = [e for e in ex_rows if e["group"] == g]
    for c in range(8):
        ax = axes[r, c]; ax.axis("off")
        if c < len(sub):
            e = sub[c]
            ax.imshow(c100.data[raw_idx[g][e["index"]]])
            ax.set_title(f"{e['unknown_class']} → {e['predicted_known_class']}\nu={e['score_mls']:.2f} (τ={tau:.2f})", fontsize=6)
fig.suptitle("Most confidently accepted unknowns (top: near, bottom: far), Vanilla + MLS", fontsize=9)
fig.tight_layout()
save_figure(fig, figure_path("task4_failure_examples.png"))

,group,unknown_class,accepted_frac,top_absorbing_known
0,near,bus,0.86,"truck (70), automobile (14), ship (1)"
1,near,camel,0.52,"deer (16), dog (12), bird (10)"
2,near,fox,0.71,"cat (31), deer (22), dog (12)"
3,near,leopard,0.55,"cat (24), frog (21), deer (4)"
4,near,motorcycle,0.39,"automobile (24), truck (8), airplane (5)"
5,near,pickup_truck,0.91,"automobile (61), truck (28), airplane (2)"
6,near,tractor,0.54,"truck (47), ship (4), automobile (3)"
7,near,wolf,0.78,"cat (37), dog (37), deer (2)"
8,far,bottle,0.34,"cat (13), ship (9), truck (6)"
9,far,bowl,0.52,"cat (14), frog (10), automobile (8)"


,group,index,unknown_class,predicted_known_class,score_mls,threshold
0,near,75,fox,dog,-12.8074,-5.8614
1,near,614,pickup_truck,automobile,-11.8478,-5.8614
2,near,346,bus,automobile,-11.6411,-5.8614
3,near,629,wolf,cat,-11.4218,-5.8614
4,near,735,leopard,cat,-11.2840,-5.8614
5,near,776,tractor,truck,-10.6780,-5.8614
6,near,468,camel,bird,-10.4511,-5.8614
7,near,270,motorcycle,truck,-10.1602,-5.8614
8,far,617,bottle,cat,-11.6861,-5.8614
9,far,704,mushroom,dog,-11.0551,-5.8614


PosixPath('/workspace/pa1-beyond-iid/report/figures/task4_failure_examples.png')

## Research questions (answer in your own words)
1. Semantic similarity and rejection; which classes are accepted and which CIFAR-10 labels absorb them → `step6_vanilla_mls_acceptance_by_class`, `task4_acceptance_heatmap`, `task4_failure_examples`.
2. What MSP / MLS / Energy / Mahalanobis capture; agreements & disagreements → `step2_vanilla_scores`, `step2_score_spearman`, `step2_score_decision_disagreement`.
3. GCSC: CSA vs. OSR, near vs. far → `step6_model_comparison`, `step6_all_scores_all_models`.
4. PROSER: rejection gain vs. CSA trade-off; placeholder score vs. MLS; RPL (optional) → `step6_model_comparison`.